# Career Agent — User Profile Extraction Pipeline

**Pipeline:** resume file (PDF/text) → raw text → LLM extraction → schema-validated structured profile → MongoDB storage.

**Setup before running:**
```
pip install anthropic pydantic pdfplumber pymongo
export ANTHROPIC_API_KEY=sk-...
export MONGODB_URI="mongodb+srv://..."
```
Or set them directly in the first code cell below.

In [1]:
# 0. Setup — install deps if needed (uncomment to run)
# !pip install anthropic pydantic pdfplumber pymongo -q

import os

# Set these directly here if you're not using shell env vars.
# os.environ["ANTHROPIC_API_KEY"] = "sk-..."
# os.environ["MONGODB_URI"] = "mongodb+srv://..."


## 1. Schema

Single source of truth for what a "user profile" looks like. Every downstream
agent (Eligibility, Relevance, etc.) reads from this shape, so it's defined
once here and reused everywhere.

In [2]:
from __future__ import annotations
from typing import List, Optional, Literal
from pydantic import BaseModel, Field, ValidationError


class RoleHeld(BaseModel):
    title: Optional[str] = None
    company: Optional[str] = None
    duration_months: Optional[float] = None
    skills_used: List[str] = Field(default_factory=list)


class Personal(BaseModel):
    name: Optional[str] = None
    location: Optional[str] = None
    location_preference: Optional[Literal["remote", "hybrid", "onsite", "any"]] = "any"
    visa_status: Optional[str] = None  # free text: "citizen", "needs sponsorship", etc.


class Experience(BaseModel):
    years_total: Optional[float] = None
    years_relevant: Optional[float] = None
    current_role: Optional[str] = None
    seniority_level: Optional[Literal["intern", "entry", "mid", "senior", "unspecified"]] = "unspecified"
    roles_held: List[RoleHeld] = Field(default_factory=list)


class Education(BaseModel):
    degree: Optional[str] = None
    field: Optional[str] = None
    institution: Optional[str] = None
    graduation_year: Optional[int] = None
    status: Optional[Literal["completed", "in_progress", "unspecified"]] = "unspecified"


class Skills(BaseModel):
    primary: List[str] = Field(default_factory=list)     # strong, demonstrated skills
    secondary: List[str] = Field(default_factory=list)   # mentioned but less central
    learning: List[str] = Field(default_factory=list)    # explicitly "currently learning"


class Preferences(BaseModel):
    desired_roles: List[str] = Field(default_factory=list)
    avoid: List[str] = Field(default_factory=list)
    industries_of_interest: List[str] = Field(default_factory=list)
    min_salary: Optional[float] = None
    company_size_preference: Optional[Literal["startup", "mid", "enterprise", "any"]] = "any"


class UserProfile(BaseModel):
    personal: Personal = Field(default_factory=Personal)
    experience: Experience = Field(default_factory=Experience)
    education: Education = Field(default_factory=Education)
    skills: Skills = Field(default_factory=Skills)
    preferences: Preferences = Field(default_factory=Preferences)
    raw_resume_text: Optional[str] = None  # kept for embedding / semantic fallback

    class Config:
        extra = "forbid"  # reject unexpected fields from a malformed LLM response

print("Schema defined: UserProfile")


Schema defined: UserProfile


/tmp/ipykernel_169/2858743797.py:50: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class UserProfile(BaseModel):


### Sanity check — validate a sample payload

Quick check that the schema behaves the way we expect before wiring the LLM
into it: a well-formed profile should pass, and a broken one (bad enum value +
an unexpected field, simulating a flaky LLM output) should be rejected.

In [3]:
import json

sample = {
    "personal": {
        "name": "Rahul Sharma",
        "location": "Gurugram, India",
        "location_preference": "remote",
        "visa_status": "citizen"
    },
    "experience": {
        "years_total": 2.5,
        "years_relevant": 2.0,
        "current_role": "Backend Developer",
        "seniority_level": "mid",
        "roles_held": [
            {"title": "Backend Developer", "company": "TechCorp", "duration_months": 24,
             "skills_used": ["Python", "Django", "PostgreSQL"]}
        ]
    },
    "education": {
        "degree": "B.Tech", "field": "Computer Science", "institution": "NIT Kurukshetra",
        "graduation_year": 2022, "status": "completed"
    },
    "skills": {
        "primary": ["Python", "Django", "SQL"],
        "secondary": ["Docker", "AWS"],
        "learning": ["Kubernetes"]
    },
    "preferences": {
        "desired_roles": [], "avoid": [], "industries_of_interest": [],
        "min_salary": None, "company_size_preference": "any"
    },
    "raw_resume_text": "sample resume text..."
}

profile = UserProfile.model_validate(sample)
print("Valid payload accepted.")
print(json.dumps(profile.model_dump(exclude={"raw_resume_text"}), indent=2))


Valid payload accepted.
{
  "personal": {
    "name": "Rahul Sharma",
    "location": "Gurugram, India",
    "location_preference": "remote",
    "visa_status": "citizen"
  },
  "experience": {
    "years_total": 2.5,
    "years_relevant": 2.0,
    "current_role": "Backend Developer",
    "seniority_level": "mid",
    "roles_held": [
      {
        "title": "Backend Developer",
        "company": "TechCorp",
        "duration_months": 24.0,
        "skills_used": [
          "Python",
          "Django",
          "PostgreSQL"
        ]
      }
    ]
  },
  "education": {
    "degree": "B.Tech",
    "field": "Computer Science",
    "institution": "NIT Kurukshetra",
    "graduation_year": 2022,
    "status": "completed"
  },
  "skills": {
    "primary": [
      "Python",
      "Django",
      "SQL"
    ],
    "secondary": [
      "Docker",
      "AWS"
    ],
    "learning": [
      "Kubernetes"
    ]
  },
  "preferences": {
    "desired_roles": [],
    "avoid": [],
    "industries_of_i

In [4]:
# Deliberately broken payload — should be rejected
bad = {
    "personal": {"location_preference": "space_station"},  # invalid enum
    "extra_field": "should not exist"                       # not in schema
}
try:
    UserProfile.model_validate(bad)
    print("ERROR: this should have failed validation")
except ValidationError as e:
    print("Correctly rejected malformed input:")
    print(e)


Correctly rejected malformed input:
2 validation errors for UserProfile
personal.location_preference
  Input should be 'remote', 'hybrid', 'onsite' or 'any' [type=literal_error, input_value='space_station', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error
extra_field
  Extra inputs are not permitted [type=extra_forbidden, input_value='should not exist', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden


## 2. Parse resume — PDF/text to raw text

Simple, dependency-light extraction. If you later need scanned/image-based
PDFs, add OCR (`pytesseract`) as a fallback when `pdfplumber` returns
near-empty text.

In [5]:
from pathlib import Path


def extract_text(file_path: str) -> str:
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"No file at {file_path}")

    if path.suffix.lower() == ".pdf":
        return _extract_from_pdf(path)
    elif path.suffix.lower() in (".txt", ".md"):
        return path.read_text(encoding="utf-8", errors="ignore")
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")


def _extract_from_pdf(path: Path) -> str:
    import pdfplumber

    text_chunks = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text_chunks.append(page.extract_text() or "")

    text = "\n".join(text_chunks).strip()

    if len(text) < 50:
        raise ValueError(
            "Extracted text is suspiciously short — this PDF may be scanned/image-based. "
            "OCR (pytesseract) would be needed to handle it."
        )
    return text

print("extract_text() ready")


extract_text() ready


## 3. Extract profile — LLM call + schema validation + retry loop

Rules baked into the prompt:
- Extract only what's explicitly stated — never guess/estimate.
- Resumes usually won't state `preferences` — leave those empty unless clearly stated.
- Output must be pure JSON matching the schema (no markdown fences, no commentary).

If the model's output fails JSON parsing or schema validation, we retry once
with the error appended to the prompt — a cheap, effective self-correction loop.

In [6]:
from anthropic import Anthropic

MODEL = "claude-sonnet-4-6"

SYSTEM_PROMPT = """You extract structured career-profile data from resume text.

Rules:
- Extract ONLY what is explicitly stated or very directly implied in the text.
- Never guess or estimate a number (like years of experience) that isn\'t stated
  or trivially computable from stated dates. If unclear, use null.
- Do not invent skills, roles, or preferences not present in the text.
- "preferences" (desired_roles, avoid, industries_of_interest, min_salary,
  company_size_preference) will usually NOT be in a resume — leave them empty
  unless the text clearly states them (e.g. an objective/summary line).
- Output ONLY a single JSON object matching the schema below. No markdown
  fences, no commentary, no preamble.

Schema:
{schema}
"""

USER_TEMPLATE = """Resume text:
---
{resume_text}
---

Return the JSON object now."""


def _client() -> Anthropic:
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        raise RuntimeError("ANTHROPIC_API_KEY is not set.")
    return Anthropic(api_key=api_key)


def _call_llm(resume_text: str, retry_note: str | None = None) -> str:
    client = _client()
    schema_json = json.dumps(UserProfile.model_json_schema(), indent=2)

    user_content = USER_TEMPLATE.format(resume_text=resume_text)
    if retry_note:
        user_content += f"\n\nYour previous output was invalid: {retry_note}\nFix it and return valid JSON only."

    response = client.messages.create(
        model=MODEL,
        max_tokens=2000,
        system=SYSTEM_PROMPT.format(schema=schema_json),
        messages=[{"role": "user", "content": user_content}],
    )
    return "".join(block.text for block in response.content if block.type == "text")


def _clean_json_string(raw: str) -> str:
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        if raw.startswith("json"):
            raw = raw[4:]
    return raw.strip()


def extract_profile(resume_text: str, max_retries: int = 1) -> UserProfile:
    last_error = None
    for attempt in range(max_retries + 1):
        raw_output = _call_llm(resume_text, retry_note=last_error)
        cleaned = _clean_json_string(raw_output)

        try:
            data = json.loads(cleaned)
        except json.JSONDecodeError as e:
            last_error = f"Output was not valid JSON: {e}"
            continue

        try:
            profile = UserProfile.model_validate(data)
            profile.raw_resume_text = resume_text
            return profile
        except ValidationError as e:
            last_error = f"JSON did not match schema: {e}"
            continue

    raise RuntimeError(f"Failed after {max_retries + 1} attempts. Last error: {last_error}")

print("extract_profile() ready")


extract_profile() ready


## 4. MongoDB storage

Upsert by `user_id`. An `embedding` field is accepted separately (not
computed here) since embedding generation is a distinct concern — you'll
likely add it once you wire up semantic retrieval over `raw_resume_text`.

In [7]:
from datetime import datetime, timezone
from pymongo import MongoClient

DB_NAME = "career_agent"
PROFILES_COLLECTION = "user_profiles"


def get_client() -> MongoClient:
    uri = os.environ.get("MONGODB_URI")
    if not uri:
        raise RuntimeError("MONGODB_URI is not set.")
    return MongoClient(uri)


def get_profiles_collection():
    client = get_client()
    return client[DB_NAME][PROFILES_COLLECTION]


def save_profile(profile: UserProfile, user_id: str, embedding: list | None = None) -> str:
    collection = get_profiles_collection()
    doc = profile.model_dump()
    doc["user_id"] = user_id
    doc["updated_at"] = datetime.now(timezone.utc)
    if embedding is not None:
        doc["resume_embedding"] = embedding

    result = collection.update_one(
        {"user_id": user_id},
        {"$set": doc, "$setOnInsert": {"created_at": datetime.now(timezone.utc)}},
        upsert=True,
    )
    if result.upserted_id:
        return str(result.upserted_id)
    existing = collection.find_one({"user_id": user_id}, {"_id": 1})
    return str(existing["_id"])


def get_profile(user_id: str) -> UserProfile | None:
    collection = get_profiles_collection()
    doc = collection.find_one({"user_id": user_id})
    if not doc:
        return None
    for key in ("_id", "user_id", "created_at", "updated_at", "resume_embedding"):
        doc.pop(key, None)
    return UserProfile.model_validate(doc)

print("Mongo storage functions ready")


Mongo storage functions ready


## 5. Run end-to-end

Set `RESUME_PATH` to a real resume file (`.pdf` or `.txt`) and run this cell.
Requires `ANTHROPIC_API_KEY` and `MONGODB_URI` to be set (cell 0).

In [8]:
RESUME_PATH = "resume.pdf"   # <- change this to your file
USER_ID = "user_123"

def run(file_path: str, user_id: str):
    print(f"[1/3] Reading resume: {file_path}")
    resume_text = extract_text(file_path)
    print(f"      Extracted {len(resume_text)} characters.")

    print("[2/3] Extracting structured profile via LLM...")
    profile = extract_profile(resume_text)
    print("      Extraction succeeded and passed schema validation.")

    print("[3/3] Saving profile to MongoDB...")
    doc_id = save_profile(profile, user_id=user_id)
    print(f"      Saved. user_id={user_id} doc_id={doc_id}")

    print("\n--- Extracted profile ---")
    print(json.dumps(profile.model_dump(exclude={"raw_resume_text"}), indent=2))
    return profile

# Uncomment to run against a real resume file + live API/DB:
# profile = run(RESUME_PATH, USER_ID)


## Next step

Once this is working against a real resume, the next notebook builds the
**job ingestion pipeline** (Adzuna + Greenhouse/Lever → normalized job
documents in MongoDB) so there's real data to match this profile against.